In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


BASE_DIR = Path.cwd().parent
BASE_DIR

DATA_DIR = BASE_DIR/"data"/"processed"
MODELS_DIR = BASE_DIR /"models"
parent_dir

'/media/ashfaque/datas/ML-projects/retail-forecast-system'

In [4]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 


In [5]:
train = pd.read_parquet(DATA_DIR/'train_filtered_ca1.parquet')
test  = pd.read_parquet(DATA_DIR/'test_filtered_ca1.parquet')



In [50]:
### Analysing the experiment results calculated
import json 

results_dir = BASE_DIR /'results/Experiments'

lgbm_direct = results_dir/'lgbm_direct_1.0yr.json'
lgbm_recursive= results_dir/'lgbm_recursive_1.0yr.json'

lgbm_direct_2 = results_dir/'lgbm_direct_2.0yr.json'
lgbm_recursive_2 = results_dir/'lgbm_recursive_2.0yr.json'

expt_1 = [lgbm_recursive,lgbm_direct]

In [134]:

def compare_expts(recursive_file,direct_file):
    """
    Extracts only monthly holding cost entries from the FVA list.
    
    Parameters:
        data (list or dict): The raw experiment output.
        
    Returns:
        pd.DataFrame: A DataFrame containing filtered holding cost FVA entries.
    """
    with open(recursive_file,'r') as f1:
        rec_data = json.load(f1)
    with open(direct_file,'r') as f2:
        direct_data = json.load(f2)

    to_rename = ['lgbm','fva_model_vs_naive']
    merge_on = ['window_id','train_start','train_end','metric','seasonal_naive',
                   'moving_average','fva_moving_average_vs_naive']

    rec_fva = pd.DataFrame(rec_data[0].get('fva')).rename(columns= {**{col : f'{col}_recursive' for col in to_rename}})
    direc_fva = pd.DataFrame(direct_data[0].get('fva')).rename(columns= {**{col : f'{col}_direct' for col in to_rename}})

    selected_cols = ['window_id','train_start','train_end','model','MAE','BIAS%','wrmsse']

    rec_metric = pd.DataFrame(rec_data[0].get('metrics'))[selected_cols]
    direc_metric = pd.DataFrame(direct_data[0].get('metrics'))[selected_cols]

    
    # pivot to table 
    pivot_table = lambda df : df.pivot(
                index=["window_id", "train_start", "train_end"],
                columns="model",
            ).stack(level=0, future_stack=True).reset_index()

    rec_metric_table = pivot_table(rec_metric).rename(columns={'lgbm':'lgbm_recursive','level_3':'metrics'})
    direc_metric_table = pivot_table(direc_metric).rename(columns={'lgbm':'lgbm_direct','level_3':'metrics'})

    merge_metric_on = ['window_id','train_start','train_end','metrics','seasonal_naive','moving_average']

    metric_merge = rec_metric_table.merge(direc_metric_table,on=merge_metric_on,how='left')


    agg_metrics = metric_merge.groupby('metrics')[model_cols].agg(['mean', 'median', 'std'])

    from src.metrics import fva 

    mean_metrics = metric_merge.groupby('metrics')[model_cols].agg('mean')
    fva_df = mean_metrics.copy()
    fva_df['moving_average_vs_naive'] = fva(mean_metrics['seasonal_naive'],mean_metrics['moving_average'])
    fva_df['recursive_vs_naive'] = fva(mean_metrics['seasonal_naive'],mean_metrics['lgbm_recursive'])
    fva_df['direct_vs_naive'] = fva(mean_metrics['seasonal_naive'],mean_metrics['lgbm_direct'])

    return agg_metrics, fva_df
    # return pd.DataFrame(metric),pd.DataFrame(fva_records) 



In [138]:
agg_metrics, fva_df = compare_expts(lgbm_recursive,lgbm_direct)

agg_metrics_2 , fva_df_2 = compare_expts(lgbm_recursive_2,lgbm_direct_2)

In [139]:
agg_metrics

model   lgbm_recursive                     moving_average                      \
                  mean    median       std           mean    median       std   
metrics                                                                         
BIAS%         2.502898  2.292658  6.549726       2.427934  2.325526  6.610428   
MAE           2.039391  2.048617  0.102713       1.246350  1.225663  0.051749   
wrmsse        0.865069  0.863952  0.052874       0.919914  0.910257  0.061345   

model   seasonal_naive                     lgbm_direct                      
                  mean    median       std        mean    median       std  
metrics                                                                     
BIAS%         1.187552  1.717401  6.768065   -1.257955 -1.099751  6.050727  
MAE           1.405214  1.432619  0.071852    1.149125  1.146889  0.055322  
wrmsse        1.185407  1.182145  0.045196    0.864037  0.854348  0.052177

In [140]:
agg_metrics_2

model   lgbm_recursive                     moving_average                      \
                  mean    median       std           mean    median       std   
metrics                                                                         
BIAS%         4.616080  5.556591  7.556540       1.900868 -0.920827  6.829988   
MAE           2.047501  2.068582  0.122857       1.243298  1.225176  0.053789   
wrmsse        0.895042  0.884616  0.070355       0.908034  0.897270  0.050607   

model   seasonal_naive                     lgbm_direct                      
                  mean    median       std        mean    median       std  
metrics                                                                     
BIAS%         0.547656  0.790705  6.983210   -0.680189  1.257845  6.211374  
MAE           1.414222  1.450476  0.073106    1.167852  1.171478  0.054504  
wrmsse        1.196807  1.192011  0.067097    0.867698  0.872825  0.050977

In [141]:
fva_df_2

model,lgbm_recursive,moving_average,seasonal_naive,lgbm_direct,moving_average_vs_naive,recursive_vs_naive,direct_vs_naive
metrics,,,,,,,
BIAS%,4.616080,1.900868,0.547656,-0.680189,-247.09,-742.88,224.20
MAE,2.047501,1.243298,1.414222,1.167852,12.09,-44.78,17.42
wrmsse,0.895042,0.908034,1.196807,0.867698,24.13,25.21,27.50


In [142]:
fva_df

model,lgbm_recursive,moving_average,seasonal_naive,lgbm_direct,moving_average_vs_naive,recursive_vs_naive,direct_vs_naive
metrics,,,,,,,
BIAS%,2.502898,2.427934,1.187552,-1.257955,-104.45,-110.76,205.93
MAE,2.039391,1.246350,1.405214,1.149125,11.31,-45.13,18.22
wrmsse,0.865069,0.919914,1.185407,0.864037,22.40,27.02,27.11


In [69]:
Deployment_dir = MODELS_DIR /'deployments/deployment_1788773717'



forecasts = pd.read_parquet(Deployment_dir/'forecasts.parquet')
inventory_costs = pd.read_parquet(Deployment_dir/'inventory_costs.parquet')
inventory_policy = pd.read_parquet(Deployment_dir/'inventory_policy.parquet')


In [5]:
# comparison with 
forecast_comparison  = pd.read_parquet(Deployment_dir/'forecast_comparison.parquet')
inventory_policy_comparison = pd.read_parquet(Deployment_dir/'inventory_policy_comparison.parquet')
inventory_cost_comparison = pd.read_parquet(Deployment_dir/'inventory_cost_comparison.parquet')


In [8]:
inventory_cost_comparison

,item_id,rmse_tau,mae_tau,n_obs,forecast_tau,safety_stock_rmse,safety_stock_mae,order_up_to_rmse,order_up_to_mae,raw_demand_std,...,protection_end_date,cycle_demand_R,cycle_stock,avg_on_hand_rmse,monthly_holding_cost_rmse,avg_on_hand_mae,monthly_holding_cost_mae,avg_on_hand_classical,monthly_holding_cost_classical,model
0,FOODS_1_001,2.828687,2.505070,18,9.261779,4.653191,4.120841,13.914970,13.382620,1.077433,...,2016-01-16,6.031775,3.015888,7.669078,1.073671,7.136728,0.999142,8.894199,1.245188,lgbm
1,FOODS_1_002,1.873357,1.535078,18,5.114355,3.081673,2.525203,8.196028,7.639559,0.689638,...,2016-01-16,3.550888,1.775444,4.857117,0.679996,4.300647,0.602091,5.538001,0.775320,lgbm
2,FOODS_1_003,4.760875,4.228223,18,8.630641,7.831639,6.955426,16.462280,15.586068,1.140808,...,2016-01-16,6.227014,3.113507,10.945146,1.532320,10.068933,1.409651,9.337582,1.307262,lgbm
3,FOODS_1_005,19.795138,15.113437,18,16.810702,32.563002,24.861603,49.373704,41.672305,1.631536,...,2016-01-16,12.642655,6.321327,38.884330,5.443806,31.182931,4.365610,15.222741,2.131184,lgbm
4,FOODS_1_006,5.385523,4.233985,18,17.175556,8.859185,6.964906,26.034741,24.140461,1.728662,...,2016-01-16,10.360256,5.180128,14.039313,1.965504,12.145034,1.700305,14.611445,2.045602,lgbm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10831,HOUSEHOLD_2_509,2.505549,1.944444,18,5.000000,4.121629,3.198611,9.121629,8.198611,0.900655,...,2016-01-30,2.000000,1.000000,5.121629,0.717028,4.198611,0.587806,5.913835,0.827937,seasonal_naive
10832,HOUSEHOLD_2_511,3.480102,3.111111,18,0.000000,5.724768,5.117778,5.724768,5.117778,1.401590,...,2016-01-30,0.000000,0.000000,5.724768,0.801468,5.117778,0.716489,7.646863,1.070561,seasonal_naive
10833,HOUSEHOLD_2_512,2.357023,1.777778,18,4.000000,3.877302,2.924444,7.877302,6.924444,1.110556,...,2016-01-30,2.000000,1.000000,4.877302,0.682822,3.924444,0.549422,7.059026,0.988264,seasonal_naive
10834,HOUSEHOLD_2_514,2.236068,2.000000,18,3.000000,3.678332,3.290000,6.678332,6.290000,0.499433,...,2016-01-30,3.000000,1.500000,5.178332,0.724966,4.790000,0.670600,4.224831,0.591476,seasonal_naive


In [9]:
from src.utils_visuals import * 


inventory_cost_model = inventory_cost_comparison[inventory_cost_comparison['model']=='lgbm']



In [10]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from config import PIPELINE_CONFIG
from src.utils_visuals import (
    plot_item_forecast_and_inventory,
    plot_inventory_policy_bars,
)

# Set this explicitly to the deployment you want to show.
artifact_dir = Path("models/deployments/deployment_1788773717")

forecasts = pd.read_parquet(Deployment_dir / "forecast_comparison.parquet")
policies = pd.read_parquet(Deployment_dir / "inventory_policy_comparison.parquet")
costs = pd.read_parquet(Deployment_dir / "inventory_cost_comparison.parquet")
test_actuals = pd.read_parquet(Deployment_dir / "test_actuals.parquet")

# Compatibility for artifacts created before actual_sales was standardized.
if "actual_sales" not in forecasts.columns:
    actuals = test_actuals.rename(columns={"sales": "actual_sales"})
    forecasts = (
        forecasts.drop(columns=["real_sales"], errors="ignore")
        .merge(
            actuals[["item_id", "dept_id", "cat_id", "date", "actual_sales"]],
            on=["item_id", "dept_id", "cat_id", "date"],
            how="left",
        )
    )

item_id = "FOODS_1_001"  # choose any deployed SKU
models = ["lgbm", "moving_average", "seasonal_naive"]

# Historical context + known test actuals
train = pd.read_parquet(PIPELINE_CONFIG["train_data_path"])
history = (
    train.loc[train["item_id"] == item_id, ["date", "sales"]]
    .sort_values("date")
    .tail(90)
)

item_forecasts = forecasts.query("item_id == @item_id").copy()
test_sales = (
    item_forecasts[["date", "actual_sales"]]
    .drop_duplicates()
    .rename(columns={"actual_sales": "sales"})
)

raw_sales = (
    pd.concat([history, test_sales], ignore_index=True)
    .drop_duplicates("date", keep="last")
    .sort_values("date")
)

forecast_lines = {
    model: item_forecasts.loc[
        item_forecasts["model"] == model, ["date", "sales_pred"]
    ]
    for model in models
}

# Optional ML quantile band; works whether quantiles were configured or not.
lgb = item_forecasts.query("model == 'lgbm'").copy()
has_quantiles = (
    {"q10", "q90"}.issubset(lgb.columns)
    and lgb["q10"].notna().any()
    and lgb["q90"].notna().any()
)

p10 = lgb[["date", "q10"]].rename(columns={"q10": "p10"}) if has_quantiles else None
p90 = lgb[["date", "q90"]].rename(columns={"q90": "p90"}) if has_quantiles else None
p95 = (
    lgb[["date", "q95"]].rename(columns={"q95": "p95"})
    if has_quantiles and "q95" in lgb.columns and lgb["q95"].notna().any()
    else None
)

fig_forecast = plot_item_forecast_and_inventory(
    raw_sales=raw_sales,
    forecasted_demand=forecast_lines,
    p10=p10,
    p90=p90,
    p95=p95,
    inventory_values=None,  # Inventory is shown separately: it is tau-day, not daily demand.
    item_id=item_id,
)
fig_forecast.show()

In [11]:
item_policy = (
    policies.query("item_id == @item_id")
    .sort_values("review_date")
    .groupby("model", as_index=False)
    .first()
)

item_cost = (
    costs.query("item_id == @item_id")
    .sort_values("review_date")
    .groupby("model", as_index=False)
    .first()
)

policy_levels = {
    row["model"]: {
        "Forecast τ demand": row["forecast_tau"],
        "Safety stock (RMSE)": row["safety_stock_rmse"],
        "Order-up-to (RMSE)": row["order_up_to_rmse"],
    }
    for _, row in item_policy.iterrows()
}

fig_policy = plot_inventory_policy_bars(
    policy_levels,
    metric_name="Units",
    item_id=item_id,
)
fig_policy.show()

holding_cost = {
    row["model"]: row["monthly_holding_cost_rmse"]
    for _, row in item_cost.iterrows()
}

fig_holding = plot_inventory_policy_bars(
    holding_cost,
    metric_name="Estimated holding cost per review period",
    item_id=item_id,
)
fig_holding.show()

In [12]:
portfolio_cost = (
    costs.groupby("model", as_index=False)["monthly_holding_cost_rmse"]
    .sum()
    .rename(columns={"monthly_holding_cost_rmse": "estimated_holding_cost"})
)

fig_portfolio = px.bar(
    portfolio_cost,
    x="model",
    y="estimated_holding_cost",
    text_auto=".2f",
    title="Estimated Policy-Implied Holding Cost Across Test Horizon",
    labels={
        "model": "Forecast model",
        "estimated_holding_cost": "Estimated holding cost",
    },
)
fig_portfolio.show()